# Mecanismos de Atenção
Notebook de estudo — evolução passo a passo do mecanismo de atenção:
1. Atenção simplificada (sem pesos treináveis)
2. Self-attention com pesos treináveis (Q, K, V)
3. Atenção causal (mascarada) + dropout
4. Multi-head attention

In [1]:
import torch
import torch.nn as nn

## 1. Atenção simplificada (sem pesos treináveis)
Ideia base: o vetor de contexto de um token é a média ponderada dos demais
tokens da sequência, onde o peso vem da similaridade (produto escalar) entre eles.

In [2]:
inputs = torch.tensor(
    [[0.43, 0.15, 0.89],
     [0.55, 0.87, 0.66],
     [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33],
     [0.77, 0.25, 0.10],
     [0.05, 0.80, 0.55]]
)

Como exemplo, usamos o token de índice 1 (`journey`, no exemplo do livro) como *query*
e calculamos sua similaridade com todos os demais tokens via produto escalar.

In [3]:
query = inputs[1]

attn_scores_2 = torch.empty(inputs.shape[0])
for i, x_i in enumerate(inputs):
    attn_scores_2[i] = torch.dot(query, x_i)

attn_scores_2

tensor([0.9544, 1.4950, 1.4754, 0.8434, 0.7070, 1.0865])

Normalizamos os scores para que somem 1. Duas formas: divisão simples pela soma,
ou (melhor, pois evita pesos negativos e tem gradientes mais estáveis) `softmax`.

In [4]:
attn_weights_2_simple = attn_scores_2 / attn_scores_2.sum()
attn_weights_2_simple, attn_weights_2_simple.sum()

(tensor([0.1455, 0.2278, 0.2249, 0.1285, 0.1077, 0.1656]), tensor(1.0000))

In [5]:
attn_weights_2 = torch.softmax(attn_scores_2, dim=0)
attn_weights_2, attn_weights_2.sum()

(tensor([0.1385, 0.2379, 0.2333, 0.1240, 0.1082, 0.1581]), tensor(1.))

O vetor de contexto do token 1 é a soma dos demais tokens ponderada pelos pesos de atenção.

In [6]:
context_vec_2 = torch.zeros(query.shape[0])
for i, x_i in enumerate(inputs):
    context_vec_2 += attn_weights_2[i] * x_i

context_vec_2

tensor([0.4419, 0.6515, 0.5683])

Generalizando para **todos** os tokens de uma vez (matriz de scores 6x6),
em vez de repetir o processo token a token com loops.

In [7]:
attn_scores = inputs @ inputs.T
attn_weights = torch.softmax(attn_scores, dim=-1)
context_vecs = attn_weights @ inputs
context_vecs

tensor([[0.4421, 0.5931, 0.5790],
        [0.4419, 0.6515, 0.5683],
        [0.4431, 0.6496, 0.5671],
        [0.4304, 0.6298, 0.5510],
        [0.4671, 0.5910, 0.5266],
        [0.4177, 0.6503, 0.5645]])

## 2. Self-attention com pesos treináveis (Q, K, V)
Agora introduzimos três matrizes de pesos treináveis — `W_query`, `W_key`, `W_value` —
que projetam cada token de entrada em três representações distintas.

In [8]:
x_2 = inputs[1]
d_in = inputs.shape[1]   # dimensão de entrada
d_out = 2                # dimensão de saída (embeddings menores para fins didáticos)

In [9]:
torch.manual_seed(123)

W_query = nn.Parameter(torch.rand(d_in, d_out))
W_key = nn.Parameter(torch.rand(d_in, d_out))
W_value = nn.Parameter(torch.rand(d_in, d_out))

In [10]:
query_2 = x_2 @ W_query
query_2

tensor([0.4306, 1.4551], grad_fn=<SqueezeBackward4>)

In [11]:
keys = inputs @ W_key
values = inputs @ W_value
keys.shape, values.shape

(torch.Size([6, 2]), torch.Size([6, 2]))

O score de atenção entre a query do token 2 e todas as keys é o produto escalar entre elas.
Escalamos pela raiz de `d_k` (dimensão das keys) para estabilizar os gradientes — essa é a
ideia da *scaled dot-product attention*.

In [12]:
attn_scores_2 = query_2 @ keys.T

d_k = keys.shape[-1]
attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
attn_weights_2

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820],
       grad_fn=<SoftmaxBackward0>)

In [13]:
context_vec_2 = attn_weights_2 @ values
context_vec_2

tensor([0.3061, 0.8210], grad_fn=<SqueezeBackward4>)

Encapsulamos essa lógica em um módulo reutilizável `SelfAttentionV1`, usando `nn.Linear`
no lugar de `nn.Parameter` manual (mais idiomático e com inicialização melhor).

In [14]:
class SelfAttentionV1(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(123)
sa_v1 = SelfAttentionV1(d_in, d_out)
sa_v1(inputs)

tensor([[-0.5337, -0.1051],
        [-0.5323, -0.1080],
        [-0.5323, -0.1079],
        [-0.5297, -0.1076],
        [-0.5311, -0.1066],
        [-0.5299, -0.1081]], grad_fn=<MmBackward0>)

## 3. Atenção causal (mascarada) + dropout
Em um modelo autoregressivo (tipo GPT), cada token só pode "olhar" para tokens
anteriores (ou ele mesmo) — nunca para o futuro. Isso é feito com uma **máscara causal**:
zeramos (via `-inf` antes do softmax) todos os scores acima da diagonal principal.

In [15]:
queries = sa_v1.W_query(inputs)
keys = sa_v1.W_key(inputs)
attn_scores = queries @ keys.T

context_length = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(context_length, context_length))
mask_simple

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])

A forma numericamente correta é aplicar a máscara **antes** do softmax, preenchendo
as posições futuras com `-inf` (assim o softmax as leva para 0 automaticamente).

In [16]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
masked_scores = attn_scores.masked_fill(mask.bool(), -torch.inf)
masked_scores

tensor([[0.3111,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.1655, 0.2602,   -inf,   -inf,   -inf,   -inf],
        [0.1667, 0.2602, 0.2577,   -inf,   -inf,   -inf],
        [0.0510, 0.1080, 0.1064, 0.0643,   -inf,   -inf],
        [0.1415, 0.1875, 0.1863, 0.0987, 0.1121,   -inf],
        [0.0476, 0.1192, 0.1171, 0.0731, 0.0477, 0.0966]],
       grad_fn=<MaskedFillBackward0>)

In [17]:
attn_weights = torch.softmax(masked_scores / keys.shape[-1]**0.5, dim=-1)
attn_weights

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.4833, 0.5167, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3190, 0.3408, 0.3402, 0.0000, 0.0000, 0.0000],
        [0.2445, 0.2545, 0.2542, 0.2468, 0.0000, 0.0000],
        [0.1994, 0.2060, 0.2058, 0.1935, 0.1953, 0.0000],
        [0.1624, 0.1709, 0.1706, 0.1654, 0.1625, 0.1682]],
       grad_fn=<SoftmaxBackward0>)

Durante o treino, aplicamos **dropout** sobre os pesos de atenção (após o softmax)
para reduzir overfitting, zerando aleatoriamente uma fração dos pesos e reescalando o resto.

In [18]:
torch.manual_seed(123)
dropout = nn.Dropout(0.5)
dropout(attn_weights)

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.6380, 0.6816, 0.6804, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.5090, 0.5085, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4120, 0.0000, 0.3869, 0.0000, 0.0000],
        [0.0000, 0.3418, 0.3413, 0.3308, 0.3249, 0.0000]],
       grad_fn=<MulBackward0>)

Encapsulamos tudo em um módulo `CausalAttention`, que também suporta **batches**
(múltiplas sequências processadas ao mesmo tempo) e registra a máscara como `buffer`
(não é parâmetro treinável, mas se move junto com o modelo entre CPU/GPU).

In [19]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        attn_scores = queries @ keys.transpose(1, 2)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / self.d_out**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = attn_weights @ values
        return context_vec

In [20]:
torch.manual_seed(123)

batch = torch.stack((inputs, inputs), dim=0)  # simula um batch com 2 sequências
context_length = batch.shape[1]

ca = CausalAttention(d_in, d_out, context_length, dropout=0.0)
context_vecs = ca(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 2]),
 tensor([[[-0.4519,  0.2216],
          [-0.5874,  0.0058],
          [-0.6300, -0.0632],
          [-0.5675, -0.0843],
          [-0.5526, -0.0981],
          [-0.5299, -0.1081]],
 
         [[-0.4519,  0.2216],
          [-0.5874,  0.0058],
          [-0.6300, -0.0632],
          [-0.5675, -0.0843],
          [-0.5526, -0.0981],
          [-0.5299, -0.1081]]], grad_fn=<UnsafeViewBackward0>))

## 4. Multi-Head Attention
Em vez de uma única "cabeça" de atenção, usamos várias em paralelo — cada uma aprende
a focar em relações diferentes entre os tokens — e concatenamos os resultados.

### 4.1 Versão simples (wrapper)
A forma mais direta: instanciar várias `CausalAttention` independentes e concatenar as saídas.

In [21]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)

In [22]:
torch.manual_seed(123)

context_length = batch.shape[1]
mha_wrapper = MultiHeadAttentionWrapper(d_in, d_out=2, context_length=context_length, dropout=0.0, num_heads=2)
context_vecs = mha_wrapper(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 4]),
 tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
          [-0.5874,  0.0058,  0.5891,  0.3257],
          [-0.6300, -0.0632,  0.6202,  0.3860],
          [-0.5675, -0.0843,  0.5478,  0.3589],
          [-0.5526, -0.0981,  0.5321,  0.3428],
          [-0.5299, -0.1081,  0.5077,  0.3493]],
 
         [[-0.4519,  0.2216,  0.4772,  0.1063],
          [-0.5874,  0.0058,  0.5891,  0.3257],
          [-0.6300, -0.0632,  0.6202,  0.3860],
          [-0.5675, -0.0843,  0.5478,  0.3589],
          [-0.5526, -0.0981,  0.5321,  0.3428],
          [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>))

### 4.2 Versão eficiente (usada em GPT-like models)
A wrapper acima roda cada cabeça em sequência, o que desperdiça paralelismo.
Na prática, usamos **uma única** matriz de projeção linear para todas as cabeças de uma vez,
e depois "dividimos" (`view`) a dimensão de saída em `num_heads` fatias — muito mais eficiente
e é assim que a atenção é implementada em modelos como o GPT-2.

In [23]:
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out precisa ser divisível por num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)  # combina as saídas das cabeças
        self.dropout = nn.Dropout(dropout)
        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, num_tokens, d_in = x.shape

        queries = self.W_query(x)
        keys = self.W_key(x)
        values = self.W_value(x)

        # divide a dimensão d_out em (num_heads, head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(
            self.mask.bool()[:num_tokens, :num_tokens], -torch.inf
        )
        attn_weights = torch.softmax(attn_scores / self.head_dim**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        return context_vec

In [24]:
torch.manual_seed(123)

context_length = batch.shape[1]
d_out = 2
mha = MultiHeadAttention(d_in, d_out, context_length, dropout=0.0, num_heads=2)
context_vecs = mha(batch)
context_vecs.shape, context_vecs

(torch.Size([2, 6, 2]),
 tensor([[[0.3190, 0.4858],
          [0.2943, 0.3897],
          [0.2856, 0.3593],
          [0.2693, 0.3873],
          [0.2639, 0.3928],
          [0.2575, 0.4028]],
 
         [[0.3190, 0.4858],
          [0.2943, 0.3897],
          [0.2856, 0.3593],
          [0.2693, 0.3873],
          [0.2639, 0.3928],
          [0.2575, 0.4028]]], grad_fn=<ViewBackward0>))

Pronto — mecanismo de atenção completo: da versão simplificada sem pesos até a
*multi-head causal self-attention* usada em modelos GPT-like. Próximo passo natural
(capítulo 4 do livro) é usar esse bloco de atenção dentro de um bloco Transformer
completo, com camadas de feed-forward, normalização e conexões residuais.